In [1]:
import pandas as pd
from urllib.parse import urlsplit
import sqlite3
import numpy as np

In [2]:
conn = sqlite3.connect("../../db/telemetry.db")

In [ ]:
df = pd.read_sql_query(
    """SELECT os.event_id, os.title, os.executable, os.event_start_time AS os_event_start_time, os.event_end_time AS os_event_end_time,
            browser.url, browser.title AS browser_tab_title, browser.event_start_time AS browser_event_start_time, browser.event_end_time AS browser_event_end_time,
            os.previous_events
        FROM os_events
        AS os LEFT JOIN browser_events AS browser
        ON os.event_id=browser.os_event_id
        LIMIT 1000;""",
    conn,
    parse_dates = ["os_event_start_time", "os_event_end_time"]
)
df.head()

,event_id,title,executable,os_event_start_time,os_event_end_time,url,browser_tab_title,browser_event_start_time,browser_event_end_time,previous_events
0,7019c0d1-4b1a-4dc3-88c9-8e67962be134,Settings,ApplicationFrameHost.exe,2026-10-13 15:53:08,2026-10-13 15:53:09,None,None,NaT,NaT,[]
1,2c42d2e6-56ac-425b-926c-d71c59736f01,C:\Windows\System32\cmd.exe,WindowsTerminal.exe,2026-10-13 15:53:09,2026-10-13 15:53:11,None,None,NaT,NaT,"[{""executable"": ""ApplicationFrameHost.exe"", ""t..."
2,478d71e1-276d-4272-9313-d09c3cecd787,.env - Personal-Telemetry - Visual Studio Code,Code.exe,2026-10-13 15:53:11,2026-10-13 15:53:17,None,None,NaT,NaT,"[{""executable"": ""WindowsTerminal.exe"", ""title""..."
3,e8ce7022-6dcc-4157-8ae4-13b82b497dd0,My_Desktop - Desktop Viewer,Citrix.DesktopViewer.App.exe,2026-10-13 15:53:17,2026-10-13 15:57:19,None,None,NaT,NaT,"[{""executable"": ""Code.exe"", ""title"": "".env - P..."
4,a0619702-55e3-4ee1-82bc-e9d85d4bc5e4,.env - Personal-Telemetry - Visual Studio Code,Code.exe,2026-10-13 15:57:19,2026-10-13 15:57:23,None,None,NaT,NaT,"[{""executable"": ""Citrix.DesktopViewer.App.exe""..."


In [44]:
df[df["url"].notna()]

,event_id,title,executable,os_event_start_time,os_event_end_time,url,browser_tab_title,browser_event_start_time,browser_event_end_time,previous_events


In [82]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   event_id                  168 non-null    object        
 1   title                     168 non-null    object        
 2   executable                168 non-null    object        
 3   os_event_start_time       168 non-null    datetime64[ns]
 4   os_event_end_time         168 non-null    datetime64[ns]
 5   url                       0 non-null      object        
 6   browser_tab_title         0 non-null      object        
 7   browser_event_start_time  0 non-null      datetime64[ns]
 8   browser_event_end_time    0 non-null      datetime64[ns]
 9   previous_events           168 non-null    object        
dtypes: datetime64[ns](4), object(6)
memory usage: 13.3+ KB


In [83]:
df.describe()

,os_event_start_time,os_event_end_time,browser_event_start_time,browser_event_end_time
count,168,168,0,0
mean,2026-10-13 17:14:45.642857216,2026-10-13 17:15:36.148809472,NaT,NaT
min,2026-10-13 15:53:08,2026-10-13 15:53:09,NaT,NaT
25%,2026-10-13 16:39:12,2026-10-13 16:39:25.750000128,NaT,NaT
50%,2026-10-13 17:31:15,2026-10-13 17:31:36,NaT,NaT
75%,2026-10-13 17:59:24.500000,2026-10-13 17:59:33.500000,NaT,NaT
max,2026-10-13 18:09:47,2026-10-13 18:14:33,NaT,NaT


In [84]:
df['executable'].value_counts()

executable
Telegram.exe                    45
Code.exe                        37
chrome.exe                      28
Citrix.DesktopViewer.App.exe    28
Citrix.EngineUI.App.exe         11
dcmsghandler.exe                 8
SnippingTool.exe                 4
explorer.exe                     3
ApplicationFrameHost.exe         1
WindowsTerminal.exe              1
SelfService.exe                  1
SearchHost.exe                   1
Name: count, dtype: int64

# Clean the data

In [85]:
df = df[~df['title'].isna()]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   event_id                  168 non-null    object        
 1   title                     168 non-null    object        
 2   executable                168 non-null    object        
 3   os_event_start_time       168 non-null    datetime64[ns]
 4   os_event_end_time         168 non-null    datetime64[ns]
 5   url                       0 non-null      object        
 6   browser_tab_title         0 non-null      object        
 7   browser_event_start_time  0 non-null      datetime64[ns]
 8   browser_event_end_time    0 non-null      datetime64[ns]
 9   previous_events           168 non-null    object        
dtypes: datetime64[ns](4), object(6)
memory usage: 13.3+ KB


In [86]:
df[(df['executable'] != 'chrome.exe') & (~df['url'].isna())]

,event_id,title,executable,os_event_start_time,os_event_end_time,url,browser_tab_title,browser_event_start_time,browser_event_end_time,previous_events


In [87]:
df[~df['url'].isna()]['executable'].value_counts()


Series([], Name: count, dtype: int64)

# Feature Engineering

### Constants

In [88]:
BROWSER_EXECUTABLES = [
    'chrome.exe'
]

### Utility functions

In [ ]:

def get_time_information(df):
    df['week_day'] = df["os_event_start_time"].dt.weekday
    df['hour'] = df["os_event_start_time"].dt.hour
    df["duration_seconds"] = (
        df["os_event_end_time"] - df["os_event_start_time"]
    ).dt.total_seconds()

    return df

def parse_dates(df):
    df['os_event_start_time'] = pd.to_datetime(df['os_event_start_time'], format='%Y-%m-%d')
    df['os_event_end_time'] = pd.to_datetime(df['os_event_end_time'], format='%Y-%m-%d')


In [90]:
df = get_time_information(df)
df["domain"] = df['url'].apply(lambda url: urlsplit(url).hostname) 
df['domain'].value_counts()

Series([], Name: count, dtype: int64)

In [91]:
df['is_os_event'] = df['executable'].isin(BROWSER_EXECUTABLES)
df = df.sort_values('is_os_event', ascending=False)
df.head()

,event_id,title,executable,os_event_start_time,os_event_end_time,url,browser_tab_title,browser_event_start_time,browser_event_end_time,previous_events,week_day,hour,duration_seconds,domain,is_os_event
91,f7cf557f-4eda-49f3-a4c6-70453bd598df,Model Telemetry Data - Google Chrome,chrome.exe,2026-10-13 17:36:08,2026-10-13 17:36:10,None,None,NaT,NaT,"[{""executable"": ""Telegram.exe"", ""title"": ""\u20...",1,17,2.0,None,True
18,d018c36f-eb0e-412d-b7e5-a3e5d768d757,Untitled - Google Chrome,chrome.exe,2026-10-13 16:06:26,2026-10-13 16:06:42,None,None,NaT,NaT,"[{""executable"": ""Telegram.exe"", ""title"": ""(7) ...",1,16,16.0,None,True
153,7a9941ab-bcfe-4367-8aaa-5841bffb40e2,Model Telemetry Data - Google Chrome,chrome.exe,2026-10-13 18:05:37,2026-10-13 18:05:49,None,None,NaT,NaT,"[{""executable"": ""Code.exe"", ""title"": ""\u25cf c...",1,18,12.0,None,True
79,b17aa2a9-b1ba-4eb3-941c-9cef80e77196,Model Telemetry Data - Google Chrome,chrome.exe,2026-10-13 17:25:20,2026-10-13 17:28:03,None,None,NaT,NaT,"[{""executable"": ""Code.exe"", ""title"": ""\u25cf c...",1,17,163.0,None,True
74,009b8823-d8e7-4910-a0a9-d3d51dbfba1d,Model Telemetry Data - Google Chrome,chrome.exe,2026-10-13 17:09:36,2026-10-13 17:10:59,None,None,NaT,NaT,"[{""executable"": ""dcmsghandler.exe"", ""title"": ""...",1,17,83.0,None,True


In [95]:
import json
previous_events_df = df[['event_id', 'previous_events']]
previous_events_df['previous_events'] = previous_events_df['previous_events'].apply(json.loads)
previous_events_df = previous_events_df.explode('previous_events')
display(previous_events_df['previous_events'].iloc[0])

# Extracting the columns from the previous events 
previous_events_df['previous_executable'] = previous_events_df['previous_events'].apply(lambda x: x.get('executable') if type(x)!=float else x)
previous_events_df['previous_publisher'] = previous_events_df['previous_events'].apply(lambda x: x.get('publisher') if type(x)!=float else x)
previous_events_df['previous_event_start_time'] = previous_events_df['previous_events'].apply(lambda x: x.get('event_start_time') if type(x)!=float else x)
previous_events_df['previous_event_end_time'] = previous_events_df['previous_events'].apply(lambda x: x.get('event_end_time') if type(x)!=float else x)

previous_events_df = previous_events_df.rename(columns={"previous_event_start_time": "os_event_start_time"})
previous_events_df = previous_events_df.rename(columns={"previous_event_end_time": "os_event_end_time"})
previous_events_df = get_time_information(previous_events_df)
# previous_events_df = (
#     previous_events_df.groupby(
#         'event_id'
#     )
#     .agg(
#         previous_events_num_unique_executables=('previous_executable', 'nunique'),
#         previous_events_num_unique_publishers=('previous_publishers', 'nunique'),

#     )
# )


previous_events_df.head()


C:\Users\yhulivatyi\AppData\Local\Temp\ipykernel_27160\1319336687.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  previous_events_df['previous_events'] = previous_events_df['previous_events'].apply(json.loads)


{'executable': 'Telegram.exe',
 'title': '\u200eYura Hulivatyi @ \u200eЮра Гуліватий (34752)',
 'publisher': 'Telegram FZ-LLC',
 'category': 'os',
 'event_start_time': '2026-10-13 17:36:08',
 'event_end_time': '2026-10-13 17:36:08'}

AttributeError: Can only use .dt accessor with datetimelike values

In [93]:
df_flat = df
df_grouped = (
    df.groupby('event_id')
    .agg(
        title=('title', 'first'),
        executable=('executable', 'first'),
        event_start_time=('os_event_start_time', 'first'),
        event_end_time=('os_event_end_time', 'first'),
        previous_events=('previous_events', 'first'),
        week_day=('week_day', 'first'),
        hour=('hour', 'first'),
        num_unique_domains=('domain', 'nunique'),
        num_unique_urls=('url', 'nunique'),
        num_browser_events=('url','size'),
        duration_seconds=('duration_seconds', 'sum'),

    )
)
df_grouped.head()

,title,executable,event_start_time,event_end_time,previous_events,week_day,hour,num_unique_domains,num_unique_urls,num_browser_events,duration_seconds
event_id,,,,,,,,,,,
0069a28f-3e57-4858-9da2-7a1cb5905be9,● classifier_training.ipynb - Personal-Telemet...,Code.exe,2026-10-13 16:31:25,2026-10-13 16:31:26,"[{""executable"": ""Citrix.DesktopViewer.App.exe""...",1,16,0,0,1,1.0
009b8823-d8e7-4910-a0a9-d3d51dbfba1d,Model Telemetry Data - Google Chrome,chrome.exe,2026-10-13 17:09:36,2026-10-13 17:10:59,"[{""executable"": ""dcmsghandler.exe"", ""title"": ""...",1,17,0,0,1,83.0
010ddbd0-864a-44d4-84fb-35082792f13f,‎Yura Hulivatyi @ ‎Юра Гуліватий (34752),Telegram.exe,2026-10-13 17:36:08,2026-10-13 17:36:08,"[{""executable"": ""chrome.exe"", ""title"": ""Model ...",1,17,0,0,1,0.0
04db45f1-19e0-48b3-a6c3-6e6dfae0ee90,Model Telemetry Data - Google Chrome,chrome.exe,2026-10-13 18:06:05,2026-10-13 18:06:38,"[{""executable"": ""Code.exe"", ""title"": ""\u25cf c...",1,18,0,0,1,33.0
083de9c7-dbd0-4b38-9b58-ebc206e3c47d,Model Telemetry Data - Google Chrome,chrome.exe,2026-10-13 17:56:10,2026-10-13 17:56:30,"[{""executable"": ""Code.exe"", ""title"": ""\u25cf c...",1,17,0,0,1,20.0


In [ ]:
df_grouped = 